# Stage 00 — Corpus scoping and tactic survey

**Track A (Buse) · Stage 0 of 10**

| | |
|---|---|
| **Input** | A pile of candidate papers, and an opinion about what the assistant is for |
| **Output** | A bounded corpus, a written domain brief, and a tactic ledger that drives stages 01-09 |
| **Promotes to** | Nothing in `src/`. This stage produces documents and a manifest, not code. |
| **Blocks** | Everything. Stage 05 (eval set) is unwritable without the brief. |

## Why this stage is first

Every later choice (chunk size, embedding model, what counts as a relevant hit)
is only answerable relative to "what is this thing supposed to answer?".
Skip it and stage 06 measures a number that means nothing, and the gate in
stage 09 protects a threshold nobody can defend.

## Part A — The domain brief

Fill in the five answers below, then run the next cell to write
`docs/domain_brief_B.md`. Keep it under a page. It is a decision record, not an essay.

1. **Topic boundary.** One sentence on what the corpus covers, one on what it
   deliberately excludes. Exclusions matter more than inclusions.
2. **Question shapes.** Three to five *shapes* of question the assistant must answer.
   Not example questions yet, shapes. For example: "compare method A and B on
   metric M", "what dataset did paper X evaluate on", "what limitation does
   approach Y state about itself".
3. **Out of scope.** Question shapes you are explicitly not serving. This is what
   stops scope creep from turning into bad eval numbers.
4. **Why this corpus.** Why these papers can actually answer those shapes. If a
   shape needs information no paper contains, either the shape or the corpus is wrong.
5. **Size and cutoff.** How many papers, what year range, what venues.

### Design choice: how big should the corpus be?

| Option | Papers | Pros | Cons |
|---|---|---|---|
| Tiny | 10-20 | Fast iteration, you can read every chunk | Retrieval is trivial, metrics saturate, tactics become indistinguishable |
| **Small (recommended)** | **40-80** | **Big enough that ranking matters, small enough to hand-label** | **A day of labelling in stage 05** |
| Medium | 200+ | Realistic | Hand-labelling becomes infeasible, so you end up trusting an LLM judge you never validated |

Recommendation: **60 papers, one coherent subtopic**. The binding constraint is not
ingestion speed, it is that *you* hand-label relevance in stage 05. A reranker cannot
be evaluated on labels you did not have time to make carefully.

In [ ]:
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

BRIEF = {
    "topic_boundary": "TODO: one sentence on what is in.",
    "excludes": "TODO: one sentence on what is deliberately out.",
    "question_shapes": [
        "TODO shape 1",
        "TODO shape 2",
        "TODO shape 3",
    ],
    "out_of_scope_shapes": [
        "TODO: a shape we are NOT serving, and why",
    ],
    "why_this_corpus": "TODO: why these papers can answer those shapes.",
    "size_target": 60,
    "year_range": "TODO e.g. 2019-2025",
    "venues": ["TODO"],
}

In [ ]:
# Writes docs/domain_brief_B.md. Re-run whenever you change BRIEF above.
lines = ["# Domain brief - Track A (Buse)", ""]
lines += ["## Scope", "", BRIEF["topic_boundary"], "", "**Excluded:** " + BRIEF["excludes"], ""]
lines += ["## Question shapes the assistant must answer", ""]
lines += [f"{i}. {s}" for i, s in enumerate(BRIEF["question_shapes"], 1)]
lines += ["", "## Explicitly out of scope", ""]
lines += [f"- {s}" for s in BRIEF["out_of_scope_shapes"]]
lines += ["", "## Why this corpus fits", "", BRIEF["why_this_corpus"], ""]
lines += ["## Corpus bounds", "",
          f"- Target size: {BRIEF['size_target']} papers",
          f"- Years: {BRIEF['year_range']}",
          f"- Venues: {', '.join(BRIEF['venues'])}", ""]
lines += ["## Consequences for later stages", "",
          "- Stage 05 eval queries are sampled across the shapes above, not invented ad hoc.",
          "- A retrieval miss on an out-of-scope shape is not a bug.",
          "- Citation precision is measured against page ranges of these papers only.", ""]

out = REPO / "docs" / "domain_brief_B.md"
# The written brief is the source of truth now. Refuse to overwrite it with this template.
if out.exists() and out.stat().st_size > 0:
    print("not overwriting", out, "- edit the markdown directly, or delete it to regenerate")
else:
    out.write_text("\n".join(lines), encoding="utf-8")
    print("wrote", out)

## Part B — The tactic ledger

This is the part you asked about: how to go into the literature and come back with
the tactic that gives the best results, rather than the tactic that was written
about most confidently.

### The one rule

> **A paper's claim is a hypothesis. Your held-out set from stage 05 is what decides.**

Nothing is adopted because a paper reported a gain. It is adopted because it beat
your current baseline on your corpus, measured in stage 06. This matters because
retrieval tactics are strongly corpus-dependent. A chunking strategy that wins on
short web passages routinely loses on dense two-column PDFs with real section
structure, which is exactly what you have.

### The adoption pipeline

Every tactic moves through five states, and the ledger records where each one is.

| State | Meaning | What you record |
|---|---|---|
| `observed` | You read about it | Source, the claim, the corpus it was measured on |
| `shortlisted` | Plausible for our corpus | Why it might transfer, estimated cost |
| `prototyped` | Implemented behind a config flag | Which notebook, which config key |
| `measured` | Run against the held-out set | Delta on nDCG@5, MRR, recall@20, plus latency |
| `adopted` / `rejected` | Decision made | The delta that justified it |

The gap between `observed` and `measured` is where most projects go wrong. They adopt
at `observed`, and then cannot explain their own pipeline in the write-up.

### What to search for, stage by stage

Do not search "best RAG pipeline". Search per stage. That is how the literature is
organised, and a per-stage ablation is the only kind you can afford to run.

| Our stage | Tactic families worth surveying | Search terms that find them |
|---|---|---|
| 01 Parsing | Layout-aware PDF extraction, table and formula handling, section detection | `scientific document layout parsing`, `PDF to structured text academic`, `section segmentation scholarly` |
| 02 Chunking | Fixed vs recursive vs section-aware vs semantic vs proposition-level; sentence-window; parent-document / small-to-big; contextual chunk prefixes; late chunking | `chunking strategy retrieval augmented generation`, `proposition retrieval`, `contextual retrieval`, `late chunking` |
| 03 Embedding | Asymmetric query/passage prefixes, size vs latency, domain adaptation, dimension truncation | `text embedding benchmark retrieval`, `asymmetric dense retrieval`, `matryoshka embeddings` |
| 04 Hybrid retrieval | Sparse-dense fusion, reciprocal rank fusion vs score normalisation, query expansion, hypothetical document embeddings, multi-query, metadata filtering | `hybrid sparse dense retrieval fusion`, `reciprocal rank fusion`, `query expansion dense retrieval`, `HyDE` |
| 05 Eval set | Pooling for relevance judgments, graded vs binary labels, annotator agreement, validating an LLM judge against human labels | `test collection construction pooling`, `graded relevance judgments`, `LLM as judge agreement human` |
| 06 Metrics | nDCG and MRR properties, recall ceilings, attribution precision | `evaluation metrics ranked retrieval`, `attribution evaluation generated text` |
| 07-08 Reranking | Cross-encoder reranking, listwise LLM reranking, distillation from a large ranker, preference-based training | `cross-encoder reranking passage`, `listwise LLM reranker`, `preference optimization ranking` |

Shortlist **two** candidates per family, not ten. Two lets you run a real A/B in the
matching notebook. Ten means none of them get measured.

### How to read a retrieval paper for this purpose

Read in this order, stop early when it fails a check. Ten minutes per paper.

1. **What corpus did they measure on?** Short web passages, or long structured
   documents? If it is not document-shaped like yours, the reported gain is weak evidence.
2. **What baseline did they beat?** A gain over pure dense retrieval with no reranker
   says nothing about a gain over hybrid plus a cross-encoder. Your baseline after
   stage 04 is already reasonably strong, which kills most reported gains.
3. **What does it cost?** Extra model calls per query, extra index build time, extra
   serve latency. A two-point nDCG gain that triples query latency is a rejection,
   and the ledger should record that reasoning.
4. **Can it live behind a flag?** If adopting it means rewriting stages 01-04, it is
   not a tactic, it is a different project. Note it and move on.
5. **What is its failure mode?** Every tactic has one. Semantic chunking fails on
   papers with weak topical drift. Query expansion fails on precise entity lookups.
   Write it down, then look for it during stage 06 error analysis.

Only then record it in the ledger.

In [ ]:
# Tactic ledger. Add a row as you read. `notebook` says where it gets measured.
# Keep `state` honest: nothing is "adopted" before stage 06 produces a delta.

TACTICS = [
    dict(
        tactic="Section-aware chunking",
        stage="02",
        state="prototyped",
        source="TODO source + year",
        claim="Respecting section boundaries beats fixed windows on structured documents",
        measured_on="TODO corpus used by the source",
        cost="none at query time, more parse logic",
        failure_mode="Papers with unlabelled headings silently fall back to fixed windows",
        config_key="chunk.strategy",
        notebook="02_chunking_B.ipynb",
        delta_ndcg5=None,
        verdict="",
    ),
    dict(
        tactic="Reciprocal rank fusion for sparse+dense",
        stage="04",
        state="prototyped",
        source="TODO",
        claim="Rank-based fusion beats a weighted score sum without needing calibration",
        measured_on="TODO",
        cost="negligible",
        failure_mode="Ignores score magnitude, so one very strong hit is under-rewarded",
        config_key="hybrid.fusion",
        notebook="04_hybrid_retrieval_B.ipynb",
        delta_ndcg5=None,
        verdict="",
    ),
    # dict(tactic="HyDE query expansion", stage="04", state="shortlisted", ...),
    # dict(tactic="Proposition-level chunking", stage="02", state="observed", ...),
]

import pandas as pd
pd.DataFrame(TACTICS)[["tactic", "stage", "state", "config_key", "delta_ndcg5", "verdict"]]

In [ ]:
# Render the ledger to docs/tactics_ledger_B.md so it lands in the final write-up.
cols = ["tactic", "stage", "state", "source", "claim", "measured_on", "cost",
        "failure_mode", "config_key", "notebook", "delta_ndcg5", "verdict"]
md_lines = ["# Tactic ledger - Track A (Buse)", "",
            "Every retrieval tactic considered, and the measured reason it was kept or dropped.",
            "`delta_ndcg5` is filled in by notebook 06 against the held-out set.", "",
            "| " + " | ".join(cols) + " |",
            "|" + "|".join(["---"] * len(cols)) + "|"]
for t in TACTICS:
    md_lines.append("| " + " | ".join(str(t.get(c, "") or "") for c in cols) + " |")

out = REPO / "docs" / "tactics_ledger_B.md"
out.write_text("\n".join(md_lines) + "\n", encoding="utf-8")
print("wrote", out)

## Part C — Corpus manifest

One row per paper, with the reason it is in. The `why_included` field is not
bureaucracy. At stage 05 you have to justify why a query is answerable at all, and
this is where that justification lives.

In [ ]:
import json, hashlib

RAW = REPO / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

def paper_id(path):
    return hashlib.sha1(path.name.encode("utf-8")).hexdigest()[:10]

rows = []
for pdf in sorted(RAW.glob("*.pdf")):
    rows.append(dict(
        paper_id=paper_id(pdf),
        filename=pdf.name,
        title="TODO",   # stage 01 fills this from the PDF, correct by hand where wrong
        year=None,
        venue="",
        why_included="TODO: which question shape from the brief does this paper serve?",
    ))

manifest = REPO / "data" / "corpus_manifest_B.jsonl"
with manifest.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"{len(rows)} papers in {RAW}, target from brief: {BRIEF['size_target']}")
if len(rows) < BRIEF["size_target"]:
    print("Under target. Either add papers, or lower the target in the brief and say why.")

## Exit checks

Do not start stage 01 until all four are true.

- [ ] `docs/domain_brief_B.md` exists, and a reader who has never seen the corpus
      could state what the assistant does and does not answer.
- [ ] Every question shape has at least three papers in the manifest that could answer it.
- [ ] `docs/tactics_ledger_B.md` has at least two shortlisted candidates for stages 02 and 04.
- [ ] Sude has read the brief. Her editorial rubric has to agree with your notion of
      "answerable", or the editor agent will flag drafts your retrieval considers correct.

## Open questions to revisit

- Is any question shape actually multi-hop? Those need two chunks from different
  papers, and the stage 04 candidate set is tuned for single-hop. Note it now,
  measure it in stage 06.
- Are any papers near-duplicates (preprint plus published version)? They split
  relevance labels across two chunks and quietly depress nDCG.